# Penerapan Algoritma Regresi untuk Memprediksi Jumlah Penyewaan Sepeda Berdasarkan Kondisi Lingkungan

**Proyek Akhir Data Science**

- Metode: Regression
- Dataset: Bike Sharing Dataset (UCI Machine Learning Repository)
- Target: `cnt` — jumlah total penyewaan sepeda
- Algoritma: Linear Regression, Decision Tree Regressor, dan Random Forest Regressor

> **Petunjuk:** Jalankan setiap sel secara berurutan di Google Colab. Hasil metrik dan grafik akan terisi setelah kode dijalankan. Jangan menuliskan hasil evaluasi sebelum menjalankan model.

## 1. Business Understanding

### 1.1 Latar belakang
Sistem penyewaan sepeda dapat dipengaruhi oleh waktu, musim, dan kondisi lingkungan seperti temperatur, kelembapan, kecepatan angin, serta kondisi cuaca. Prediksi jumlah penyewaan dapat membantu pengelola memperkirakan kebutuhan sepeda pada waktu tertentu.

### 1.2 Rumusan masalah
1. Bagaimana hubungan faktor waktu dan kondisi lingkungan dengan jumlah penyewaan sepeda?
2. Seberapa baik algoritma regresi memprediksi jumlah penyewaan sepeda?
3. Algoritma mana yang memberikan hasil prediksi terbaik berdasarkan MAE, RMSE, dan R²?

### 1.3 Tujuan
- Memahami pola penyewaan sepeda melalui EDA.
- Melakukan pemeriksaan dan preprocessing data.
- Melatih dan membandingkan beberapa algoritma regresi.
- Menentukan model terbaik berdasarkan data pengujian.

### 1.4 Metode dan metrik
Model yang dibandingkan adalah Linear Regression, Decision Tree Regressor, dan Random Forest Regressor. Evaluasi menggunakan:
- **MAE**: rata-rata selisih absolut prediksi dan nilai aktual; makin kecil makin baik.
- **RMSE**: akar rata-rata kuadrat kesalahan; makin kecil makin baik dan lebih sensitif terhadap kesalahan besar.
- **R²**: proporsi variasi target yang dijelaskan model; umumnya makin mendekati 1 makin baik.

Pembagian data dilakukan berdasarkan urutan tanggal: data lebih awal digunakan untuk pelatihan dan data lebih akhir untuk pengujian. Pendekatan ini lebih realistis untuk data yang memiliki urutan waktu.

## 2. Data Gathering

Dataset diambil dari UCI Machine Learning Repository.

- **Sumber:** https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset
- **Penulis dataset:** Hadi Fanaee-T
- **Periode:** 2011–2012
- **Unit observasi:** jumlah penyewaan sepeda per jam
- **Lisensi:** Creative Commons Attribution 4.0 International (CC BY 4.0)
- **Sitasi:** Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5W894

File `hour.csv` dipilih karena berisi pengamatan per jam sehingga dapat dianalisis berdasarkan jam, cuaca, dan faktor waktu.

### Deskripsi kolom yang digunakan
- `season`: musim
- `yr`: tahun (0 = 2011, 1 = 2012)
- `mnth`: bulan
- `hr`: jam
- `holiday`: indikator hari libur
- `weekday`: hari dalam minggu
- `workingday`: indikator hari kerja
- `weathersit`: kategori kondisi cuaca
- `temp`: temperatur yang telah dinormalisasi
- `atemp`: temperatur terasa yang telah dinormalisasi
- `hum`: kelembapan yang telah dinormalisasi
- `windspeed`: kecepatan angin yang telah dinormalisasi
- `cnt`: jumlah total penyewaan sepeda (target)

Kolom `casual` dan `registered` tidak digunakan sebagai fitur karena `cnt` merupakan jumlah dari kedua kolom tersebut. Menggunakannya akan menyebabkan **target leakage**. Kolom ID `instant` juga tidak digunakan sebagai prediktor.

In [ ]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

In [ ]:
# Unduh dataset resmi dari UCI
zip_url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
zip_path = "/content/bike_sharing_dataset.zip"
extract_dir = "/content/bike_sharing_dataset"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(zip_url, zip_path)

os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

csv_path = os.path.join(extract_dir, "hour.csv")
df = pd.read_csv(csv_path)

print("Dataset berhasil dimuat.")
print("Ukuran dataset:", df.shape)
display(df.head())

### 2.1 Pemeriksaan awal
Periksa bentuk dataset, nama kolom, tipe data, dan contoh data untuk memastikan file terbaca dengan benar.

In [ ]:
print("Jumlah baris dan kolom:", df.shape)
print("\nNama kolom:")
print(df.columns.tolist())

print("\nTipe data:")
display(df.dtypes.to_frame("dtype"))

print("\nInformasi dataset:")
df.info()

## 3. Exploratory Data Analysis (EDA)

EDA dilakukan untuk memahami distribusi target, pola penyewaan berdasarkan waktu, hubungan faktor lingkungan dengan target, dan kualitas data. Setiap grafik perlu dijelaskan berdasarkan hasil yang muncul ketika notebook dijalankan.

In [ ]:
# Statistik deskriptif untuk kolom numerik
display(df.describe().T)

In [ ]:
# Distribusi jumlah penyewaan sepeda
plt.figure(figsize=(9, 5))
plt.hist(df["cnt"], bins=40)
plt.title("Distribusi Jumlah Penyewaan Sepeda per Jam")
plt.xlabel("Jumlah penyewaan (cnt)")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata penyewaan berdasarkan jam
hourly_avg = df.groupby("hr", as_index=False)["cnt"].mean()

plt.figure(figsize=(10, 5))
plt.plot(hourly_avg["hr"], hourly_avg["cnt"], marker="o")
plt.title("Rata-rata Penyewaan Sepeda Berdasarkan Jam")
plt.xlabel("Jam")
plt.ylabel("Rata-rata penyewaan")
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata penyewaan berdasarkan kondisi cuaca
weather_avg = df.groupby("weathersit", as_index=False)["cnt"].mean()

plt.figure(figsize=(8, 5))
plt.bar(weather_avg["weathersit"].astype(str), weather_avg["cnt"])
plt.title("Rata-rata Penyewaan Berdasarkan Kondisi Cuaca")
plt.xlabel("Kategori cuaca (weathersit)")
plt.ylabel("Rata-rata penyewaan")
plt.tight_layout()
plt.show()

In [ ]:
# Hubungan temperatur dengan jumlah penyewaan
plt.figure(figsize=(8, 5))
plt.scatter(df["temp"], df["cnt"], alpha=0.25)
plt.title("Temperatur dan Jumlah Penyewaan Sepeda")
plt.xlabel("Temperatur ternormalisasi (temp)")
plt.ylabel("Jumlah penyewaan (cnt)")
plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata penyewaan berdasarkan musim
season_avg = df.groupby("season", as_index=False)["cnt"].mean()

plt.figure(figsize=(8, 5))
plt.bar(season_avg["season"].astype(str), season_avg["cnt"])
plt.title("Rata-rata Penyewaan Berdasarkan Musim")
plt.xlabel("Musim (1–4)")
plt.ylabel("Rata-rata penyewaan")
plt.tight_layout()
plt.show()

### Catatan interpretasi EDA
Setelah menjalankan grafik, tulis interpretasi berdasarkan angka dan pola yang benar-benar terlihat. Contoh aspek yang bisa dibahas:
- Apakah distribusi `cnt` simetris atau menceng?
- Pada jam berapa rata-rata penyewaan terlihat tinggi atau rendah?
- Apakah rata-rata penyewaan berbeda menurut kondisi cuaca atau musim?
- Apakah hubungan temperatur dengan jumlah penyewaan terlihat meningkat, menurun, atau tidak linear?

Jangan menganggap hubungan pada grafik sebagai bukti sebab-akibat.

## 4. Cleaning dan Preprocessing

Tahap ini memeriksa missing value, duplikat, tipe data, dan kemungkinan outlier. Dataset UCI ini didokumentasikan tidak memiliki missing value, tetapi pemeriksaan tetap dilakukan sebagai bagian dari prosedur kualitas data.

Outlier tidak langsung dihapus: nilai ekstrem pada jumlah penyewaan bisa merupakan kejadian yang valid. Hasil pemeriksaan perlu didiskusikan sebelum memutuskan penanganannya.

In [ ]:
# 4.1 Missing values
missing_counts = df.isna().sum().sort_values(ascending=False)
display(missing_counts.to_frame("missing_count"))

print("Total missing value:", int(df.isna().sum().sum()))

In [ ]:
# 4.2 Duplikat
duplicate_count = int(df.duplicated().sum())
print("Jumlah baris duplikat:", duplicate_count)

# Hapus hanya jika ada baris yang benar-benar identik.
df_clean = df.drop_duplicates().copy()
print("Ukuran sebelum:", df.shape)
print("Ukuran setelah drop_duplicates:", df_clean.shape)

In [ ]:
# 4.3 Periksa rentang dan tipe data
display(df_clean[["temp", "atemp", "hum", "windspeed", "cnt"]].describe().T)
print("Tipe data dteday sebelum konversi:", df_clean["dteday"].dtype)

df_clean["dteday"] = pd.to_datetime(df_clean["dteday"], errors="coerce")
print("Jumlah tanggal yang gagal dikonversi:", int(df_clean["dteday"].isna().sum()))

In [ ]:
# 4.4 Deteksi outlier dengan aturan IQR pada beberapa kolom numerik
# Ini hanya menghitung kandidat outlier, bukan langsung menghapusnya.
outlier_summary = []
for col in ["temp", "atemp", "hum", "windspeed", "cnt"]:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = int(((df_clean[col] < lower) | (df_clean[col] > upper)).sum())
    outlier_summary.append({
        "kolom": col,
        "batas_bawah_IQR": lower,
        "batas_atas_IQR": upper,
        "jumlah_kandidat_outlier": n_outliers
    })

display(pd.DataFrame(outlier_summary))

### Keputusan cleaning
- Missing value diperiksa dan jumlahnya dilaporkan.
- Baris duplikat identik dihapus jika ditemukan.
- Tanggal dikonversi ke format datetime untuk membantu pengurutan data.
- Kandidat outlier ditandai dengan aturan IQR. Nilai ekstrem tidak otomatis dihapus karena bisa menggambarkan kondisi permintaan yang nyata.
- `instant`, `dteday`, `casual`, dan `registered` tidak dipakai sebagai fitur model. `instant` adalah ID, `dteday` digunakan untuk pengurutan kronologis, sedangkan `casual` dan `registered` menimbulkan target leakage untuk target `cnt`.

## 5. Modelling

### 5.1 Pemilihan fitur
Fitur yang digunakan mencakup faktor waktu, cuaca, dan lingkungan. `season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday`, dan `weathersit` diperlakukan sebagai kategori. Variabel temperatur, kelembapan, dan kecepatan angin diperlakukan sebagai numerik.

`temp` dan `atemp` mungkin berkorelasi kuat; keduanya tetap disertakan pada eksperimen awal, tetapi hal ini dapat dibahas sebagai keterbatasan interpretasi Linear Regression.

In [ ]:
categorical_features = [
    "season", "yr", "mnth", "hr", "holiday",
    "weekday", "workingday", "weathersit"
]
numeric_features = ["temp", "atemp", "hum", "windspeed"]
feature_cols = categorical_features + numeric_features
target_col = "cnt"

# Buat data untuk modelling dan urutkan berdasarkan tanggal.
model_df = df_clean.dropna(subset=["dteday"]).sort_values("dteday").reset_index(drop=True)

X = model_df[feature_cols]
y = model_df[target_col]

# Split kronologis: 80% awal untuk train, 20% akhir untuk test.
split_index = int(len(model_df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Jumlah data train:", len(X_train))
print("Jumlah data test:", len(X_test))
print("Rentang tanggal train:", model_df["dteday"].iloc[0], "sampai", model_df["dteday"].iloc[split_index - 1])
print("Rentang tanggal test:", model_df["dteday"].iloc[split_index], "sampai", model_df["dteday"].iloc[-1])

In [ ]:
# Preprocessing: imputasi median untuk numerik dan modus untuk kategori,
# kemudian One-Hot Encoding untuk kategori.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(
        random_state=RANDOM_STATE,
        min_samples_leaf=5
    ),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=150,
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        n_jobs=-1
    )
}

fitted_models = {}
predictions = {}
results = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    fitted_models[name] = pipe
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)

## 6. Evaluation

Gunakan tabel hasil evaluasi di atas untuk membandingkan model pada data pengujian yang sama. Model dengan MAE dan RMSE lebih kecil serta R² lebih tinggi biasanya lebih baik, tetapi hasil harus dibaca bersama konteks data dan pola kesalahan.

In [ ]:
# Grafik perbandingan RMSE
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.title("Perbandingan RMSE Antar-Model")
plt.xlabel("Model")
plt.ylabel("RMSE (lebih kecil lebih baik)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Grafik nilai aktual vs prediksi dari model dengan RMSE terkecil
best_model_name = results_df.iloc[0]["Model"]
best_predictions = predictions[best_model_name]

print("Model terbaik berdasarkan RMSE:", best_model_name)

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_predictions, alpha=0.35)
min_val = min(float(np.min(y_test)), float(np.min(best_predictions)))
max_val = max(float(np.max(y_test)), float(np.max(best_predictions)))
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.title(f"Aktual vs Prediksi — {best_model_name}")
plt.xlabel("Nilai aktual cnt")
plt.ylabel("Nilai prediksi cnt")
plt.tight_layout()
plt.show()

In [ ]:
# Ringkasan metrik model terbaik
best_row = results_df.iloc[0]
print(f"Model terbaik berdasarkan RMSE: {best_row['Model']}")
print(f"MAE  : {best_row['MAE']:.3f}")
print(f"RMSE : {best_row['RMSE']:.3f}")
print(f"R²   : {best_row['R2']:.3f}")

### Panduan menulis interpretasi evaluasi
Isi interpretasi setelah menjalankan notebook:
1. Sebutkan model dengan RMSE terkecil dan bandingkan dengan model lain.
2. Jelaskan MAE dalam satuan jumlah penyewaan sepeda per jam.
3. Jelaskan R² sebagai ukuran seberapa besar variasi target yang dapat dijelaskan model pada data uji.
4. Perhatikan grafik aktual vs prediksi: apakah prediksi mengikuti garis diagonal, dan apakah model cenderung meleset pada nilai tinggi atau rendah?
5. Hindari menyebut model “akurat” hanya dari satu metrik.

## 7. Conclusion

Lengkapi kesimpulan setelah seluruh kode dijalankan. Gunakan angka hasil evaluasi aktual dari notebook, bukan angka perkiraan.

### Template kesimpulan
Berdasarkan eksperimen pada Bike Sharing Dataset, penelitian ini membandingkan Linear Regression, Decision Tree Regressor, dan Random Forest Regressor untuk memprediksi jumlah penyewaan sepeda (`cnt`) berdasarkan faktor waktu dan lingkungan. Model terbaik berdasarkan RMSE adalah **[isi nama model]**, dengan MAE **[isi nilai]**, RMSE **[isi nilai]**, dan R² **[isi nilai]** pada data pengujian.

Hasil EDA menunjukkan bahwa **[isi pola yang benar-benar terlihat dari grafik]**. Pemeriksaan kualitas data menemukan **[isi jumlah missing value, duplikat, dan kandidat outlier dari hasil pemeriksaan]**. Kandidat outlier tidak dihapus otomatis karena nilai ekstrem mungkin merupakan kondisi penyewaan yang valid.

Keterbatasan penelitian ini meliputi data yang hanya mencakup Washington, D.C. pada 2011–2012, kemungkinan perubahan pola permintaan dari waktu ke waktu, dan keterbatasan fitur yang tersedia. Model ini belum tentu langsung berlaku untuk kota atau periode lain tanpa pengujian tambahan.

## Referensi

Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5W894

UCI Machine Learning Repository. *Bike Sharing Dataset*. https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset

## Checklist sebelum dikumpulkan
- [ ] Jalankan semua sel dari atas ke bawah tanpa error.
- [ ] Baca dan interpretasikan semua grafik berdasarkan hasil aktual.
- [ ] Isi kesimpulan dengan nilai metrik yang benar-benar diperoleh.
- [ ] Pastikan tidak ada `casual` dan `registered` dalam fitur model.
- [ ] Simpan notebook yang sudah dijalankan beserta outputnya.
- [ ] Buat README repository yang menjelaskan proyek, cara menjalankan, sumber dataset, dan anggota kelompok.